# Ejercicio RSA

Gabriel Garcia - 21352

## Parte 1

**¿Por qué no conviene cifrar un documento directamente con RSA?**

### Respuesta

No conviene cifrar un documento completo directamente con RSA por varias razones.

Primero, RSA es un algoritmo más lento que los cifrados simétricos como AES. Entonces, si el archivo es grande, cifrar todo con RSA sería ineficiente y tardaría más.

Segundo, RSA no está pensado para cifrar cantidades grandes de datos. En la práctica solo puede cifrar mensajes pequeños, porque el tamaño del mensaje debe ser menor que el tamaño de la clave y además el padding también ocupa espacio.

Por eso, en sistemas reales se usa un cifrado híbrido:
- RSA se usa para proteger la clave AES
- AES se usa para cifrar el documento completo

Esto combina lo mejor de ambos:
- RSA permite intercambiar la clave de forma segura
- AES permite cifrar archivos grandes de forma rápida

Entonces, la razón principal es que RSA es más lento y no es práctico para documentos grandes, mientras que AES sí está diseñado para cifrar grandes cantidades de información.

# Parte 2: Generación de claves

In [1]:
!pip install pycryptodome

In [2]:
from Crypto.PublicKey import RSA

# Tamaño de la llave RSA
BITS = 2048

# Contraseña para proteger la clave privada
PASSPHRASE = "lab04uvg"

# Generar el par de claves
key = RSA.generate(BITS)

# Exportar la clave privada en PEM con protección
private_key = key.export_key(
    format='PEM',
    passphrase=PASSPHRASE,
    pkcs=8,
    protection='scryptAndAES128-CBC'
)

# Exportar la clave pública en PEM
public_key = key.publickey().export_key(format='PEM')

# Guardar los archivos
with open('private_key.pem', 'wb') as f:
    f.write(private_key)

with open('public_key.pem', 'wb') as f:
    f.write(public_key)

print('Claves generadas correctamente.')
print('Se creó private_key.pem')
print('Se creó public_key.pem')

Claves generadas correctamente.
Se creó private_key.pem
Se creó public_key.pem


In [3]:
# Abrir y mostrar el contenido de la clave pública
with open('public_key.pem', 'r', encoding='utf-8') as f:
    contenido_publico = f.read()

print(contenido_publico)

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAvpvuHtGO7naeZ5UE7ibe
CXeUPi4QE7hLRg/1VyBTRUz+tNIhSZWx/kFESYx3HFa3UjYHYOGPmkqKkX0Pnoie
Cc2J28S0HKyCVgYWNwfb0dXhBxLPI2WPzwmpJCO+8zaKnSjONq038NNrdem3BYsJ
up6X48sy4yqjBc6Ryts/i7Jfjxq+JDgc4KUvazC7PFGKbX2INAvMGw5qeXTxJWK0
d09Iop7jmJGt3LBESgL4PvO4y2zWBGVk6RidQIDydohc356uIBjPTW4h/VAH9Ue8
WEB+PrbgRjUg27+Je208uECygbOzy57NzGUUg/AhjDYBIYNjeTMrc9UKyAExdSBl
pQIDAQAB
-----END PUBLIC KEY-----


## ¿Qué información contiene un archivo .pem?

Un archivo .pem contiene datos criptográficos codificados en Base64 y rodeados por encabezados y pies de texto.

En este ejercicio:
- public_key.pem contiene la clave pública RSA.
- private_key.pem contiene la clave privada RSA, pero protegida con contraseña.

### Estructura de public_key.pem
Cuando se abre con un editor de texto, normalmente se observa algo así:

```text
-----BEGIN PUBLIC KEY-----
...contenido en Base64...
-----END PUBLIC KEY-----
```

### ¿Qué significa cada parte?
- BEGIN PUBLIC KEY: marca el inicio de la clave pública.
- El bloque intermedio: es la clave codificada en Base64.
- END PUBLIC KEY: marca el final del contenido.

### Descripción sencilla
El archivo PEM no guarda la clave en texto normal entendible para una persona. Lo que muestra es una representación textual segura y estándar del contenido binario de la clave, para que pueda almacenarse o compartirse fácilmente.

En una clave pública RSA, internamente se guarda información como:
- el módulo (n)
- el exponente público (e)

En la clave privada también se incluyen valores privados adicionales necesarios para descifrar y firmar.

## Parte 3: Cifrado y descifrado directo con RSA-OAEP


In [4]:
from Crypto.Cipher import PKCS1_OAEP

# Mensaje que vamos a cifrar
mensaje = b"Hola, este es un mensaje secreto usando RSA-OAEP"

# Cargar la clave publica desde el archivo PEM
with open('public_key.pem', 'rb') as f:
    public_key_data = f.read()

# Crear el objeto de clave publica
public_key = RSA.import_key(public_key_data)

# Crear el cifrador OAEP con la clave publica
cipher_rsa_encrypt = PKCS1_OAEP.new(public_key)

# Cifrar el mensaje
ciphertext = cipher_rsa_encrypt.encrypt(mensaje)

print('Mensaje original:')
print(mensaje)
print('\nMensaje cifrado en bytes:')
print(ciphertext)

Mensaje original:
b'Hola, este es un mensaje secreto usando RSA-OAEP'

Mensaje cifrado en bytes:
b'Vb^\xe2\xc3Wf\xaa\x97Q\xf5\xa1Nc\xaf\xf0\x17R\x82\xfc\xcaN("\xbd;\xf6\x0b\xf0\x00;\xa9\x96\xa6\xf2E\xecoa\x8fV\xa6:#\x1a\xf8}%\xfd\xfa\xe5\xc7+#\xc4\xce\xbe\xa9fbB7\x86\xdbO\xd6\xc7\xc0\x01He\x07\xb3Q\xc0\tB#N\xce\xa7\xb0\x88\xf6\xba\x00\x84\xb7i\xf2\x01\xe5\x0f\xd7\x10\xca\x9c\x9cwP:`\x94\xb9\x86/\xbf\xf6XFk\x96\xf4\xf1\x8e\xfd\xc5\xeddj\x9e\x05n\xfe\x1c\x06\x97\xa9m\xd6`j\xb3\xcc\x1e\x9f\xff\xec\x8f*0o\x0b\x17++\x06\xed\x97.\x1bF\xf8\xf6Y\x184\x86\x8b\x8a\x121\x9c\xcb\x11\xbcA@\xe6Dq\xde\xe8v7\xbdh\xa1\x80%\xda:\xb6\x0eAq?\xe1\xc5WJ\xfb\xd6\xbcW\xc7\xefpH/6\x8c\xbc\xfakR\xebd+"\x99~\x96\x80@\xbc\x8b\xd2\xab)#\x18!`vD\xa63\xc3h\xec\x9cN;\x8aj\xaa1}\xf5\xcc\xf3\x97\xa8\x8b\xff\xab~\xb2[\xb3\xeb(w\xd3X'


### Descifrar con la clave privada

Ahora se usa la clave privada protegida con passphrase para recuperar el mensaje original.

In [5]:
# Cargar la clave privada desde el archivo PEM
with open('private_key.pem', 'rb') as f:
    private_key_data = f.read()

# Importar la clave privada usando la contraseña
private_key = RSA.import_key(private_key_data, passphrase=PASSPHRASE)

# Crear el objeto para descifrar con OAEP
cipher_rsa_decrypt = PKCS1_OAEP.new(private_key)

# Descifrar el mensaje cifrado
mensaje_descifrado = cipher_rsa_decrypt.decrypt(ciphertext)

print('Mensaje descifrado:')
print(mensaje_descifrado)
print('\nTexto recuperado:')
print(mensaje_descifrado.decode('utf-8'))

Mensaje descifrado:
b'Hola, este es un mensaje secreto usando RSA-OAEP'

Texto recuperado:
Hola, este es un mensaje secreto usando RSA-OAEP


## cifrar el mismo mensaje dos veces

La consigna pide demostrar que al cifrar el mismo mensaje dos veces con OAEP, el resultado cambia.

In [6]:
# Crear dos cifradores separados usando la misma clave publica
cipher_1 = PKCS1_OAEP.new(public_key)
cipher_2 = PKCS1_OAEP.new(public_key)

# Cifrar el mismo mensaje dos veces
ciphertext_1 = cipher_1.encrypt(mensaje)
ciphertext_2 = cipher_2.encrypt(mensaje)

print('Ciphertext 1:')
print(ciphertext_1)
print('\nCiphertext 2:')
print(ciphertext_2)
print('\n¿Son iguales?')
print(ciphertext_1 == ciphertext_2)

Ciphertext 1:
b'^\x94\xff\r\x08\x95\x0e\xb9\xa8\xa7\xb7\x1dT2\xcai6\xae|\xebg\xa0Q\x0f\x13\xad7\xcbWv\\\xdcn\xdb\xfd\x12%\xe3:w\x00\xd9\xd3\xb7\xfdwML\xb7\xe5\xafO\xd0-68\xbc\x06\x03\xa8U=\xf6\xaa\x9e\xea\xcfuE\xca\xee\x84k&\x80_9)5x\x91Gw-\xd7\xc5\xf9\x07\xe8\xfa}\xc4\xab\xe3>\x13\x1a\xa3K\xf3\xcc\xfff\xd3\x16\xf5A~"w8p\xa1\x97;\x05\x85\x81I\x94\x9a0\xaa<o\xf5K\xb4\xb1!l\xeb\x0ek\xfb?\x85\x86\xe6jX\xc4\x1b\xb3"o\xbf\xd4\xc5\x88\x02<\xef4\\x\xe3\x8b*\xef`\xb2\t\xac\xe0j\xfa|\xf9\xd6E\xb0|&*\xe5\xdf<\x8a\xab\xb2{\x05\xeeMJ\xacMb\xd6\xb7\xae\xd6w\xab}[\x91c\x96\xeb\xbf\x0e.\xe4c\xd4\x03\x18~\xdd\x81I*\x93\xc5\x16\x81\x86p\x10\xe0\xf8\xac\xdd\xc1\x96\xb3U\xadj\xd5<8\xfd\xf9\x7f\xc8\x92\xb2\xb8\x1ez\xd8[,\xd6f\x048\x8b8\xbd\xb7\xfc\xec'

Ciphertext 2:
b'\x89A\x1fh\xe1\xa9\xdf\x03s\xd3\xb6NE@\xb2\xad\xcb\x10\xf0)o\x14\x16\xb7\xdbZr\xf8\xeb;\xd97\xe6]t\x10\x9d\xbc\xf1\x1c\xd5\xa8\xe93r\xdcY\x1f\x81\x7f$Pg\xfc\xeau\xa8\x15\x90PQ\x8e\xa6\x16\x96FK\x96\x0e\x0e\xa3\x1f\x7f\x9d"y\xf0!\xc9\xc4_t\x

## ¿Por qué cambian los resultados si el mensaje es el mismo?

Aunque el mensaje y la clave pública sean los mismos, RSA-OAEP no produce siempre el mismo ciphertext.

Esto ocurre porque OAEP agrega aleatoriedad antes de aplicar la operación RSA. En cada cifrado se usa un valor aleatorio interno llamado seed, que cambia el resultado final.

Por eso:
- mismo mensaje
- misma clave pública
- pero distinto valor aleatorio
- da como resultado ciphertext diferente

### Propiedad importante de OAEP
OAEP es un esquema de padding probabilístico. Eso significa que introduce datos aleatorios para evitar que el cifrado sea determinista.

### ¿Por qué esto es útil?
Porque si el cifrado siempre produjera el mismo resultado para el mismo mensaje, un atacante podría reconocer patrones. Con OAEP, eso no pasa fácilmente, ya que cada cifrado luce diferente aunque el contenido original sea igual.


## Parte 4 - Cifrado híbrido RSA + AES

In [7]:
documento_texto = """Contrato confidencial\nCliente: Juan Perez\nMonto: Q45,000\nCiudad: Guatemala\n"""

# Guardamos un archivo simple para usarlo como documento
with open('documento.txt', 'w', encoding='utf-8') as f:
    f.write(documento_texto)

print('Documento de prueba creado')

Documento de prueba creado


### Generar una clave AES de 256 bits

In [8]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

# 32 bytes = 256 bits
aes_key = get_random_bytes(32)

print('Clave AES generada')
print(aes_key)

Clave AES generada
b'\x85\xbah\xc3\x0c\xa6\xd8\x85\xf9\xfa\xe4\xb1\xa6\x01\xc5\x1b\xfe\xbe\xfb\x84\xf6g\x89@\x8c4\x03F\xfa\x85\x0e\xd3'


### Cifrar el documento con AES-GCM

In [9]:
# Leemos el documento en bytes
with open('documento.txt', 'rb') as f:
    documento_bytes = f.read()

# AES-GCM cifra y también genera un tag de autenticidad
cipher_aes = AES.new(aes_key, AES.MODE_GCM)
ciphertext_doc, tag = cipher_aes.encrypt_and_digest(documento_bytes)
nonce = cipher_aes.nonce

print('Nonce:')
print(nonce)
print('\nTag:')
print(tag)
print('\nCiphertext del documento:')
print(ciphertext_doc)

Nonce:
b'\x10\xf8\x8d\xf8\x15\xa2\x8d9\xea\x01\xf5\xf9\x8d\xeb\x86\x02'

Tag:
b'm\xba\xbe^J7\xce\x9fx\x15]\xd0\x0e\xfc,\xbc'

Ciphertext del documento:
b'\x9a4\x9e\xf9`\x9e\x80\x07\xf9\xa3\xa8\x84\xc3>\x9f@;\x8e\xb3y\xbf\x1b\xa3\xaa\xcd\xc8\x83\xcd\xf3YzO\x16\xeeR\xb2\xb0\x9d\x17\xab\x05\xa8fS-,#\xaf\x8d`\xc0\n\x87=\x01\xb6\xcby\xd4}(\x0fl\xf0\xe7\x8b\x06\xb2\xd3`\xe2uY\x80Wg\x07\x88U'


### Cifrar la clave AES con la clave pública RSA

In [10]:
# Usamos la clave publica para proteger la clave AES
cipher_rsa_key = PKCS1_OAEP.new(public_key)
aes_key_cifrada = cipher_rsa_key.encrypt(aes_key)

print('Clave AES cifrada con RSA:')
print(aes_key_cifrada)

Clave AES cifrada con RSA:
b'\x0c\xfe:K\xf5+\xd5\xc8\x8e\x10\xec\x9b\x04\xdf\x1e\x87\xe6\xafJ\x8e\x80\xa6\xa7\x16\x9ar\xa6\xa3&\x97\x08\xac\xe1o\x01N|_\x84\xb0A\x07UqR\xa8jQ\xc9\xbeuX\x96\xea\xe3\x90`\x81\xbeY\xdc\x81\xa4hb\xab\xa1\x1a\xe8\xcb\xb2\xa6\xe9\xfaG\x1c\x83\x1d\xa3\xfeJ\xc3\xb2\xf58\xc5\xd6"N5\x80\x1d\xe2\rd_1\x8a\xff\xb0\x95\x7f\x81\x1c\xfe\xe8\xc9\x83q\xc1\xbe\x99\xb3\xce\x12=\xe0\xa9\xfdM\xe3\xf1\x1f\xd04\xc67\x06\xd8\xbd`#\xa5|\xd0\xc7\x90w\xde\x0e\xddfS\xf0\xa6a\xee;\x87\xa3^\xb7K\xb2\x0f\x95\xaep\x03\x12\xae\r\x8dx\x87\xec\xb7lC\xd7\xe7\x1c\x13~\x06J\x83\xa8\x1d\xef\xa6\xfa\xb5\xb8\xd0\\\xdf\x0e-1\x04\x18\xed\x8e\xf6\x19\x0e\xb8DA\xaa\x89\x87\xd1N\xb9j\xf9zOt\xb6\xdf\xf4\xcbJ\xee\xf1\xd90\xd4\x94^\xbf\x82SL3\xc0\x18\xf62?/\x95\x9cK\xb1\xca^d\xc5\x83\x86\xfe/\x80\x98\x88Q\xc7\xe8\xb2\x8a\x92\xf7'


### Juntar todo lo que se enviaría

In [11]:
paquete = {
    'aes_key_cifrada': aes_key_cifrada,
    'nonce': nonce,
    'tag': tag,
    'ciphertext_doc': ciphertext_doc
}

print('Paquete listo')
print('Partes del paquete: aes_key_cifrada, nonce, tag y ciphertext_doc')

Paquete listo
Partes del paquete: aes_key_cifrada, nonce, tag y ciphertext_doc


### Descifrar para comprobar que sí funciona

In [12]:
# Primero recuperamos la clave AES usando la clave privada RSA
cipher_rsa_private = PKCS1_OAEP.new(private_key)
aes_key_recuperada = cipher_rsa_private.decrypt(paquete['aes_key_cifrada'])

# Luego usamos esa clave para descifrar el documento
cipher_aes_dec = AES.new(aes_key_recuperada, AES.MODE_GCM, nonce=paquete['nonce'])
documento_descifrado = cipher_aes_dec.decrypt_and_verify(paquete['ciphertext_doc'], paquete['tag'])

print('Documento recuperado:')
print(documento_descifrado.decode('utf-8'))

Documento recuperado:
Contrato confidencial
Cliente: Juan Perez
Monto: Q45,000
Ciudad: Guatemala



### Respuesta

En el cifrado híbrido, RSA no cifra todo el documento. RSA solo cifra la clave AES. Luego AES cifra el contenido completo del archivo.

Esto se hace porque AES es mucho más rápido y sirve para datos grandes, mientras que RSA se usa mejor para proteger claves pequeñas.

En este ejemplo:
- la clave AES de 256 bits se generó aleatoriamente
- el documento se cifró con AES-GCM
- la clave AES se cifró con RSA-OAEP
- después se recuperó la clave AES y se descifró el documento otra vez

AES-GCM además da confidencialidad e integridad, porque usa el tag para verificar que el contenido no fue alterado.